In [12]:
import torch
from torch import nn
from torchvision import models, transforms
from datasets import load_dataset
import numpy as np
import pandas as pd
import time

import bvtrain as bv
from share import hub
from share.model import build_predictor, load_bundle
from share.constants import IMG_SIZE, MEAN, STD

from diffusers import StableDiffusionImg2ImgPipeline
from ip_adapter import IPAdapter
from huggingface_hub import hf_hub_download, snapshot_download
import open_clip

env = bv.setup()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ds_test = load_dataset("dbabnigg/botanical-vision-256", split="test", streaming=True)
ds_train_full = load_dataset("dbabnigg/botanical-vision-256", split="train")  # non-streaming, needed for retrieval

CHAMPION_AUTHOR = "dbabnigg"
CHAMPION_NAME = "improved-full-t4"
ckpt_path = hub.pull(CHAMPION_AUTHOR, CHAMPION_NAME)
predict_topk = build_predictor(ckpt_path)
bundle = load_bundle(ckpt_path)
labels = bundle["labels"]

net = models.resnet50()
net.fc = nn.Linear(net.fc.in_features, len(labels))
net.load_state_dict(bundle["state_dict"])
net.eval()
net.to(device)

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(bundle.get("img_size", IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(bundle.get("mean", MEAN), bundle.get("std", STD)),
])

#fast retrieval: build species -> indices lookup once
t0 = time.time()
species_to_indices = {}
for idx, label_id in enumerate(ds_train_full["label"]):
    species_name = labels[label_id]
    species_to_indices.setdefault(species_name, []).append(idx)
print(f"Index built in {time.time() - t0:.1f}s, {len(species_to_indices)} species")

def retrieve_conspecifics(species_name, k=4):
    idxs = species_to_indices.get(species_name, [])
    if not idxs:
        return []
    chosen = np.random.choice(idxs, size=min(k, len(idxs)), replace=False)
    return [ds_train_full[int(i)]["image"] for i in chosen]

#diffusion pipeline and IP-Adapter
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

ip_ckpt = hf_hub_download(repo_id="h94/IP-Adapter", filename="models/ip-adapter_sd15.bin")
encoder_snapshot = snapshot_download(repo_id="h94/IP-Adapter", allow_patterns="models/image_encoder/*")
image_encoder_path = f"{encoder_snapshot}/models/image_encoder"
ip_model = IPAdapter(pipe, image_encoder_path, ip_ckpt, device)

IMG2IMG_STRENGTH = 0.3

# force IP-Adapter components to float32 to match the CPU pipeline
ip_model.image_encoder.to(device=device, dtype=torch.float32)
if hasattr(ip_model, "image_proj_model"):
    ip_model.image_proj_model.to(device=device, dtype=torch.float32)
for attn_processor in pipe.unet.attn_processors.values():
    if hasattr(attn_processor, "to_k_ip"):
        attn_processor.to(device=device, dtype=torch.float32)

def stylize(input_image):
    top_names = predict_topk(input_image)
    predicted_species = top_names[0]
    exemplars = retrieve_conspecifics(predicted_species)
    if not exemplars:
        return None, predicted_species, None
    generated = ip_model.generate(
        pil_image=exemplars[0],
        image=input_image,
        strength=IMG2IMG_STRENGTH,
        num_samples=1,
        num_inference_steps=20,
    )[0]
    return generated, predicted_species, exemplars[0]

#score
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model.to(device).eval()

def species_fidelity_top1(generated_image, original_species_name):
    pred_names = predict_topk(generated_image)
    return int(pred_names[0] == original_species_name)

def species_fidelity_top5(generated_image, original_species_name):
    pred_names = predict_topk(generated_image)
    return int(original_species_name in pred_names)

def clip_similarity(img_a, img_b):
    a = clip_preprocess(img_a).unsqueeze(0).to(device)
    b = clip_preprocess(img_b).unsqueeze(0).to(device)
    with torch.no_grad():
        emb_a = clip_model.encode_image(a)
        emb_b = clip_model.encode_image(b)
    return torch.cosine_similarity(emb_a, emb_b).item()

N_SAMPLE = 20
ds_test_eval = load_dataset("dbabnigg/botanical-vision-256", split="test", streaming=True)

results = []
for i, ex in enumerate(ds_test_eval):
    if i >= N_SAMPLE:
        break
    o_img = ex["image"].convert("RGB")
    o_species = ex.get("species", labels[ex["label"]])
    g_img, p_species, e_used = stylize(o_img)
    if g_img is None:
        continue
    fid1 = species_fidelity_top1(g_img, o_species)
    fid5 = species_fidelity_top5(g_img, o_species)
    sim = clip_similarity(g_img, o_img)
    results.append({"species": o_species, "fidelity_top1": fid1, "fidelity_top5": fid5, "clip_sim": sim})

results_df = pd.DataFrame(results)
print(f"N evaluated: {len(results_df)}")
print(f"Mean species-fidelity (top-1): {results_df['fidelity_top1'].mean():.3f}")
print(f"Mean species-fidelity (top-5): {results_df['fidelity_top5'].mean():.3f}")
print(f"Mean CLIP similarity: {results_df['clip_sim'].mean():.3f}")
results_df.to_csv("checkpoints/stylize_eval_results.csv", index=False)

device: cpu | data: huggingface | env: local
Building species index (one-time cost)...
Index built in 2.1s, 4094 species


100%|███████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:09<00:00,  1.64s/it]


N evaluated: 20
Mean species-fidelity (top-1): 0.000
Mean species-fidelity (top-5): 0.050
Mean CLIP similarity: 0.681
